# Bank Marketing — Model Experiments

## 1. Introduction

이 노트북에서는 Logistic Regression baseline을 재현하고, 후보 모델 비교와
hyperparameter tuning을 통해 최종 평가에 사용할 구성을 선택한다.

- 기존 feature 정책과 시간 순서에 따른 Train/Validation 분할을 유지한다.
- Notebook 02·03과 같은 sentinel-aware 전처리를 사용한다.
- Train 내부의 time-aware CV로 모델을 비교하고 유망한 계열을 tuning한다.
- Validation에서 AP, ROC-AUC, Top-K 지표와 모델 복잡도를 함께 비교한다.
- Test를 사용하기 전에 모델 구성과 최종 평가 규칙을 확정한다.

## 2. Data & Pipeline Setup

### 2.1 Load Data & Splits

원본 순서의 앞 37,069행만 읽어 Train 28,831행과 Validation 8,238행을 준비한다.
분할 경계는 전체 데이터 기준으로 유지하며 Test는 로딩하지 않는다.
Notebook 03과 동일하게 `y`와 `duration`을 제외한 19개 입력과 이진 target을 분리한다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "bank-additional" / "bank-additional-full.csv"
)

# Established boundaries from the full 41,188-row dataset.
train_end = 28831
valid_end = 37069

df = pd.read_csv(RAW_DATA_PATH, sep=";", nrows=valid_end)
train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()

In [2]:
target = "y"
excluded_features = ["duration"]
modeling_features = [
    column for column in df.columns if column not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

target_mapping = {"no": 0, "yes": 1}
y_train = train_df[target].map(target_mapping)
y_valid = valid_df[target].map(target_mapping)

pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid)],
        "features": [X_train.shape[1], X_valid.shape[1]],
        "positive_rate": [y_train.mean(), y_valid.mean()],
    },
    index=["Train", "Validation"],
)

,rows,features,positive_rate
Train,28831,19,0.055704
Validation,8238,19,0.138626


### 2.2 Sentinel-Aware Preprocessing

Notebook 02·03의 전처리를 그대로 사용한다. 일반 수치형 8개와 `pdays`에서 파생한
이전 연락 여부·경과 일수는 scaling하고, 범주형 10개는 one-hot encoding한다.
`pdays = 999`는 두 파생값 모두 `0`으로 표현하며 `"unknown"`은 범주로 유지한다.

전처리와 모델을 하나의 Pipeline으로 묶는다. 이후 CV에서는 이 Pipeline을 복제해
각 training fold에서만 scaler 통계, encoder 범주와 모델을 학습한다.

In [3]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]


def split_pdays(values):
    pdays = np.asarray(values).reshape(-1)
    previously_contacted = (pdays != 999).astype(float)
    elapsed_days = np.where(pdays == 999, 0.0, pdays.astype(float))
    return np.column_stack([previously_contacted, elapsed_days])


def pdays_feature_names(transformer, input_features):
    return np.asarray(
        ["pdays_previously_contacted", "pdays_elapsed"],
        dtype=object,
    )

In [4]:
pdays_transformer = Pipeline(
    steps=[
        (
            "derive",
            FunctionTransformer(
                split_pdays,
                validate=False,
                feature_names_out=pdays_feature_names,
            ),
        ),
        ("scale", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("pdays", pdays_transformer, ["pdays"]),
    ],
    remainder="drop",
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

## 3. Baseline Reproduction

Notebook 03과 같은 `LogisticRegression(max_iter=1000)`을 Train에서 학습하고
Validation의 양성 클래스 점수를 계산한다. AP를 주요 지표로, ROC-AUC를 보조 지표로 사용한다.

In [5]:
baseline_pipeline.fit(X_train, y_train)
y_valid_score = baseline_pipeline.predict_proba(X_valid)[:, 1]

fitted_model = baseline_pipeline.named_steps["model"]
pd.Series(
    {
        "optimization_iterations": fitted_model.n_iter_[0],
        "transformed_features": fitted_model.n_features_in_,
        "validation_scores": len(y_valid_score),
    },
    name="value",
)

optimization_iterations      35
transformed_features         62
validation_scores          8238
Name: value, dtype: int64

In [6]:
validation_prevalence = y_valid.mean()
baseline_ap = average_precision_score(y_valid, y_valid_score)
baseline_roc_auc = roc_auc_score(y_valid, y_valid_score)

general_results = pd.Series(
    {
        "validation_prevalence": validation_prevalence,
        "average_precision": baseline_ap,
        "roc_auc": baseline_roc_auc,
    },
    name="Logistic Regression",
)

general_results.to_frame().round(6)

,Logistic Regression
validation_prevalence,0.138626
average_precision,0.288234
roc_auc,0.688023


### 3.1 Top-K Ranking Metrics

상위 5%, 10%, 20%에서 `ceil`로 K를 정하고, 반올림하지 않은 점수를
내림차순 stable sort로 정렬한다. 동점은 원본 순서를 유지하며 Lift는 Validation 양성 비율을 기준으로 한다.

In [7]:
ranked_indices = np.argsort(-y_valid_score, kind="stable")
ranked_y = y_valid.to_numpy()[ranked_indices]
cumulative_positives = np.cumsum(ranked_y)
total_positives = int(y_valid.sum())

ranking_rows = []
for fraction in [0.05, 0.10, 0.20]:
    k = int(np.ceil(len(y_valid) * fraction))
    captured = int(cumulative_positives[k - 1])
    precision = captured / k

    ranking_rows.append(
        {
            "top_fraction": fraction,
            "k": k,
            "captured_subscribers": captured,
            "precision_at_k": precision,
            "recall_at_k": captured / total_positives,
            "lift_at_k": precision / validation_prevalence,
        }
    )

ranking_results = pd.DataFrame(ranking_rows)
ranking_results.round(6)

,top_fraction,k,captured_subscribers,precision_at_k,recall_at_k,lift_at_k
0,0.05,412,179,0.434466,0.156743,3.134090
1,0.10,824,309,0.375000,0.270578,2.705123
2,0.20,1648,499,0.302791,0.436953,2.184233


Baseline AP `0.288234`, ROC-AUC `0.688023`, Lift@10% `2.705123`은
Notebook 03의 저장 결과와 소수점 여섯 자리까지 일치한다. 이 결과를 후보 비교의 기준으로 사용한다.
Train 양성 비율은 5.57%, Validation은 13.86%이므로 기간별 AP를 직접 비교해 개선을 판단하지 않는다.

## 4. Model-Family Comparison

### 4.1 Hypotheses & Candidate Models

전처리를 유지하고 학습 방식이 다른 세 후보의 초기 설정을 비교한다.

| 후보 | 가설 | 초기 설정 |
|---|---|---|
| Logistic Regression | 선형 baseline이 시간 변화 속에서도 경쟁력을 유지할 수 있다. | 기존 baseline |
| Random Forest | 비선형 관계와 상호작용이 순위 성능을 개선할 수 있다. | Tree 100개, `random_state=42` |
| RBF SVM | Kernel 기반 비선형 경계가 선형 모델보다 나을 수 있다. | `C=1.0`, `gamma="scale"` |

나머지는 설치된 버전의 기본값을 사용하고 class weight는 바꾸지 않는다.
SVM은 decision-function 점수, 나머지는 양성 클래스 점수로 평가한다.
초기 설정의 비교이며 각 계열에서 얻을 수 있는 최고 성능을 의미하지 않는다.

In [8]:
from time import perf_counter

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

candidate_models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
    ),
    ("RBF SVM", SVC(kernel="rbf", C=1.0, gamma="scale")),
]

### 4.2 Time-Aware Cross-Validation

Train 안에서 학습 구간이 확장되는 `TimeSeriesSplit` fold 4개를 사용한다.
모든 후보에 같은 경계를 적용하고, 각 학습 구간은 평가 구간보다 앞선 행만 포함한다.
Fold별 표본 수와 양성 비율을 먼저 확인한다.

`gap=0`을 사용한다. 완전한 시각 정보와 고객 ID가 없어 동일한 달력상 기간이나
반복 고객 간 독립성을 보장하지 못한다. Validation과 Test는 CV에 사용하지 않는다.

In [9]:
time_cv = TimeSeriesSplit(n_splits=4)
cv_splits = list(time_cv.split(X_train))

fold_rows = []
for fold, (train_indices, eval_indices) in enumerate(cv_splits, start=1):
    fold_rows.append(
        {
            "fold": fold,
            "train_start": int(train_indices[0]),
            "train_end_exclusive": int(train_indices[-1] + 1),
            "eval_start": int(eval_indices[0]),
            "eval_end_exclusive": int(eval_indices[-1] + 1),
            "train_rows": len(train_indices),
            "eval_rows": len(eval_indices),
            "train_prevalence": y_train.iloc[train_indices].mean(),
            "eval_prevalence": y_train.iloc[eval_indices].mean(),
        }
    )

fold_summary = pd.DataFrame(fold_rows)
fold_summary

,fold,train_start,train_end_exclusive,eval_start,eval_end_exclusive,train_rows,eval_rows,train_prevalence,eval_prevalence
0,1,0,5767,5767,11533,5767,5766,0.029478,0.040062
1,2,0,11533,11533,17299,11533,5766,0.034770,0.062088
2,3,0,17299,17299,23065,17299,5766,0.043875,0.056018
3,4,0,23065,23065,28831,23065,5766,0.046911,0.090878


각 fold에서 전처리와 모델을 새로 fit하고 AP, ROC-AUC, 학습·점수 계산 시간을 기록한다.
시간에는 전처리가 포함된다. 비용 비교를 위해 후보와 fold를 순차 실행하고 Random Forest도
worker 하나를 사용한다. 실제 시간은 실행 환경의 영향을 받는다.

In [10]:
cv_rows = []
for model_name, estimator in candidate_models:
    for fold, (train_indices, eval_indices) in enumerate(cv_splits, start=1):
        X_fold_train = X_train.iloc[train_indices]
        y_fold_train = y_train.iloc[train_indices]
        X_fold_eval = X_train.iloc[eval_indices]
        y_fold_eval = y_train.iloc[eval_indices]

        fold_pipeline = Pipeline(
            steps=[
                ("preprocessor", clone(preprocessor)),
                ("model", clone(estimator)),
            ]
        )

        fit_start = perf_counter()
        fold_pipeline.fit(X_fold_train, y_fold_train)
        fit_seconds = perf_counter() - fit_start

        score_start = perf_counter()
        if model_name == "RBF SVM":
            fold_scores = fold_pipeline.decision_function(X_fold_eval)
        else:
            fold_scores = fold_pipeline.predict_proba(X_fold_eval)[:, 1]
        score_seconds = perf_counter() - score_start

        cv_rows.append(
            {
                "model": model_name,
                "fold": fold,
                "average_precision": average_precision_score(y_fold_eval, fold_scores),
                "roc_auc": roc_auc_score(y_fold_eval, fold_scores),
                "fit_seconds": fit_seconds,
                "score_seconds": score_seconds,
            }
        )
        print(f"{model_name}, fold {fold}: completed")

cv_results = pd.DataFrame(cv_rows)
cv_results.merge(
    fold_summary[["fold", "eval_rows", "eval_prevalence"]],
    on="fold",
    how="left",
).round(6)

Logistic Regression, fold 1: completed


Logistic Regression, fold 2: completed


Logistic Regression, fold 3: completed


Logistic Regression, fold 4: completed


Random Forest, fold 1: completed


Random Forest, fold 2: completed


... (3 outputs left out)

RBF SVM, fold 2: completed


RBF SVM, fold 3: completed


RBF SVM, fold 4: completed


,model,fold,average_precision,roc_auc,fit_seconds,score_seconds,eval_rows,eval_prevalence
0,Logistic Regression,1,0.040861,0.526077,0.019830,0.009294,5766,0.040062
1,Logistic Regression,2,0.072159,0.556953,0.480786,0.026232,5766,0.062088
2,Logistic Regression,3,0.057222,0.501474,0.463319,0.013033,5766,0.056018
3,Logistic Regression,4,0.162315,0.638632,0.450714,0.019539,5766,0.090878
4,Random Forest,1,0.037671,0.481446,0.250012,0.040487,5766,0.040062
5,Random Forest,2,0.061313,0.493599,0.474158,0.090389,5766,0.062088
6,Random Forest,3,0.062955,0.511292,0.734482,0.045060,5766,0.056018
7,Random Forest,4,0.103968,0.539133,1.014834,0.045579,5766,0.090878
8,RBF SVM,1,0.037099,0.462820,0.211769,0.197109,5766,0.040062
9,RBF SVM,2,0.057516,0.482071,1.290041,0.399728,5766,0.062088


### 4.3 Results & Tuning Candidates

평균 CV AP와 fold별 AP를 함께 확인한다. 표준편차는 기간에 따른 변동이며,
학습 구간이 겹치는 fold에서 계산했으므로 신뢰구간으로 해석하지 않는다.

In [11]:
cv_summary = (
    cv_results.groupby("model", sort=False)
    .agg(
        mean_ap=("average_precision", "mean"),
        std_ap=("average_precision", "std"),
        min_ap=("average_precision", "min"),
        mean_roc_auc=("roc_auc", "mean"),
        total_fit_seconds=("fit_seconds", "sum"),
        total_score_seconds=("score_seconds", "sum"),
    )
    .sort_values("mean_ap", ascending=False, kind="stable")
)
cv_summary.round(6)

,mean_ap,std_ap,min_ap,mean_roc_auc,total_fit_seconds,total_score_seconds
model,,,,,,
Logistic Regression,0.083139,0.054310,0.040861,0.555784,1.414649,0.068098
Random Forest,0.066477,0.027535,0.037671,0.506367,2.473486,0.221515
RBF SVM,0.060447,0.022258,0.037099,0.481263,24.511334,2.335867


In [12]:
fold_ap_comparison = cv_results.pivot(
    index="fold", columns="model", values="average_precision"
)
fold_ap_comparison["eval_prevalence"] = fold_summary.set_index("fold")[
    "eval_prevalence"
]
fold_ap_comparison.round(6)

model,Logistic Regression,RBF SVM,Random Forest,eval_prevalence
fold,,,,
1,0.040861,0.037099,0.037671,0.040062
2,0.072159,0.057516,0.061313,0.062088
3,0.057222,0.056453,0.062955,0.056018
4,0.162315,0.090721,0.103968,0.090878


Logistic Regression의 평균 CV AP는 `0.083139`로 Random Forest `0.066477`,
RBF SVM `0.060447`보다 높다. 4개 fold 중 3개에서 선형 모델이 앞섰고,
Random Forest는 세 번째 fold에서만 앞섰다. 초기 비선형 모델의 일관된 개선은 확인하지 못했다.

SVM은 AP가 가장 낮고 계산 비용도 높아 추가 탐색에서 제외한다.
Logistic Regression과 비선형 중 더 유망한 Random Forest를 tuning한다.
이는 현재 탐색 예산 안의 결정이며 모든 SVM 설정의 가능성을 배제하지 않는다.

Fold 양성 비율은 약 4.01~9.09%로 달라지므로 AP 변동은 양성 비율과 함께 해석한다.

## 5. Hyperparameter Tuning

### 5.1 Hypotheses & Search Scope

Logistic Regression의 정규화 강도와 Random Forest의 복잡도를 조정하면
초기 설정보다 일반화 성능이 좋아질 수 있다는 가설을 확인한다.
동일한 Train 내부 fold로 아래 범위만 탐색하며 결과를 보고 범위를 넓히지 않는다.

| 계열 | 탐색 범위 |
|---|---|
| Logistic Regression | `C = [0.01, 0.1, 1.0, 10.0]` |
| Random Forest | `max_depth = [5, 10, None]`, `min_samples_leaf = [5, 20]` |

나머지 설정은 유지한다. 코드는 초기 CV 결과에서 비선형 계열을 선택하며,
이번 실행에서는 Random Forest가 선택되어 위 탐색을 수행했다.
선택된 계열이 SVM일 때의 분기도 정의되어 있지만 이번에는 실행하지 않았다.

In [13]:
from sklearn.model_selection import GridSearchCV

nonlinear_name = cv_summary.loc[["Random Forest", "RBF SVM"]]["mean_ap"].idxmax()
nonlinear_model = next(
    estimator for name, estimator in candidate_models if name == nonlinear_name
)

lr_grid = {"model__C": [0.01, 0.1, 1.0, 10.0]}
if nonlinear_name == "Random Forest":
    nonlinear_grid = {
        "model__max_depth": [5, 10, None],
        "model__min_samples_leaf": [5, 20],
    }
else:
    nonlinear_grid = {
        "model__C": [0.1, 1.0, 10.0],
        "model__gamma": ["scale", 0.01],
    }

print("Nonlinear tuning candidate:", nonlinear_name)
print("Logistic Regression grid:", lr_grid)
print("Nonlinear grid:", nonlinear_grid)

Nonlinear tuning candidate: Random Forest
Logistic Regression grid: {'model__C': [0.01, 0.1, 1.0, 10.0]}
Nonlinear grid: {'model__max_depth': [5, 10, None], 'model__min_samples_leaf': [5, 20]}


### 5.2 Train-Only CV Search

원본 입력과 전체 Pipeline을 `GridSearchCV`에 전달해 각 설정과 fold에서 전처리를 새로 학습한다.
평균 CV AP로 설정을 정하며, `refit=False`로 탐색 직후 자동 재학습은 하지 않는다.
각 계열의 설정 하나를 정한 뒤 전체 Train에서 학습해 Validation을 비교한다.

In [14]:
lr_search = GridSearchCV(
    Pipeline(
        [
            ("preprocessor", clone(preprocessor)),
            ("model", LogisticRegression(max_iter=1000)),
        ]
    ),
    param_grid=lr_grid,
    scoring="average_precision",
    cv=cv_splits,
    refit=False,
    n_jobs=1,
    error_score="raise",
)
nonlinear_search = GridSearchCV(
    Pipeline(
        [("preprocessor", clone(preprocessor)), ("model", clone(nonlinear_model))]
    ),
    param_grid=nonlinear_grid,
    scoring="average_precision",
    cv=cv_splits,
    refit=False,
    n_jobs=1,
    error_score="raise",
)

lr_search.fit(X_train, y_train)
nonlinear_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [5, 10, ...], 'model__min_samples_leaf': [5, 20]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",False
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(5767,)), ...), (array([ 

### 5.3 Results & Candidate Configurations

설정별 평균 CV AP, 표준편차와 학습 시간을 비교한다.
같은 fold에서 여러 설정을 탐색했으므로 최고 CV 점수는 독립적인 최종 성능이 아니다.

In [15]:
search_rows = []
for name, search in [
    ("Logistic Regression", lr_search),
    (nonlinear_name, nonlinear_search),
]:
    table = pd.DataFrame(search.cv_results_)
    table["family"] = name
    search_rows.append(
        table[
            [
                "family",
                "params",
                "mean_test_score",
                "std_test_score",
                "mean_fit_time",
                "rank_test_score",
            ]
        ]
    )

tuning_results = pd.concat(search_rows, ignore_index=True)
tuning_results.sort_values(["family", "rank_test_score"], kind="stable")

,family,params,mean_test_score,std_test_score,mean_fit_time,rank_test_score
1,Logistic Regression,{'model__C': 0.1},0.085400,0.054863,0.713565,1
2,Logistic Regression,{'model__C': 1.0},0.083139,0.047033,0.449244,2
3,Logistic Regression,{'model__C': 10.0},0.082591,0.046142,0.482617,3
0,Logistic Regression,{'model__C': 0.01},0.056723,0.009318,0.324618,4
9,Random Forest,"{'model__max_depth': None, 'model__min_samples...",0.065949,0.021370,0.498241,1
8,Random Forest,"{'model__max_depth': None, 'model__min_samples...",0.065376,0.022058,0.537404,2
6,Random Forest,"{'model__max_depth': 10, 'model__min_samples_l...",0.065254,0.022814,0.424576,3
7,Random Forest,"{'model__max_depth': 10, 'model__min_samples_l...",0.063888,0.018685,0.419005,4
5,Random Forest,"{'model__max_depth': 5, 'model__min_samples_le...",0.063554,0.015382,0.353381,5
4,Random Forest,"{'model__max_depth': 5, 'model__min_samples_le...",0.063378,0.016284,0.396637,6


In [16]:
tuned_lr = clone(lr_search.estimator).set_params(**lr_search.best_params_)
tuned_nonlinear = clone(nonlinear_search.estimator).set_params(
    **nonlinear_search.best_params_
)

pd.DataFrame(
    [
        {
            "family": "Logistic Regression",
            "best_cv_ap": lr_search.best_score_,
            "parameters": lr_search.best_params_,
        },
        {
            "family": nonlinear_name,
            "best_cv_ap": nonlinear_search.best_score_,
            "parameters": nonlinear_search.best_params_,
        },
    ]
)

,family,best_cv_ap,parameters
0,Logistic Regression,0.085400,{'model__C': 0.1}
1,Random Forest,0.065949,"{'model__max_depth': None, 'model__min_samples..."


Logistic Regression은 `C=0.1`에서 CV AP `0.085400`으로 초기 `0.083139`보다 높았다.
Random Forest의 최선 설정은 `max_depth=None`, `min_samples_leaf=20`이며 CV AP는
`0.065949`이다. 초기 `0.066477`보다 낮아 제한된 tuning이 개선을 보장하지는 않았다.
이 두 설정을 Validation 후보로 넘기고 기존 baseline도 유지한다.

초기 비교 표의 표준편차는 pandas의 `ddof=1`, GridSearchCV는 `ddof=0` 기준이다.
두 표의 표준편차 숫자를 직접 비교하지 않는다.

## 6. Validation Comparison

### 6.1 AP, ROC-AUC & Top-K Metrics

Train에서 정한 두 후보와 baseline을 전체 Train에서 fit하고 같은 Validation에 평가한다.
후보별 AP, ROC-AUC와 Top-5/10/20% 지표를 비교하며 탐색 범위를 다시 넓히지 않는다.
Top-K 규칙은 baseline과 동일하다.

Test 접근 전에 정한 선택 규칙을 적용한다. 최고 AP와의 차이가 0.005 이내이면
Lift@10%를 비교하고, Lift도 상대적으로 2% 이내이면 단순한 후보를 우선한다.
과거 실험에서도 Validation을 사용했으므로 완전히 새로운 개발 평가 기간은 아니다.

In [17]:
def ranking_metrics(y_true, scores):
    ranked_y = y_true.to_numpy()[np.argsort(-scores, kind="stable")]
    cumulative_positives = np.cumsum(ranked_y)
    prevalence = y_true.mean()
    rows = []
    for fraction in [0.05, 0.10, 0.20]:
        k = int(np.ceil(len(y_true) * fraction))
        captured = int(cumulative_positives[k - 1])
        precision = captured / k
        rows.append(
            {
                "top_fraction": fraction,
                "k": k,
                "captured_subscribers": captured,
                "precision_at_k": precision,
                "recall_at_k": captured / y_true.sum(),
                "lift_at_k": precision / prevalence,
            }
        )
    return pd.DataFrame(rows)


validation_candidates = [
    ("Baseline Logistic Regression", clone(baseline_pipeline)),
    ("Tuned Logistic Regression", tuned_lr),
    (f"Tuned {nonlinear_name}", tuned_nonlinear),
]
validation_rows = []
validation_ranking_rows = []
validation_scores = {}
validation_models = {}
for name, pipeline in validation_candidates:
    fit_start = perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = perf_counter() - fit_start
    score_start = perf_counter()
    if "SVM" in name:
        scores = pipeline.decision_function(X_valid)
    else:
        scores = pipeline.predict_proba(X_valid)[:, 1]
    score_seconds = perf_counter() - score_start
    top_k = ranking_metrics(y_valid, scores)
    validation_rows.append(
        {
            "candidate": name,
            "average_precision": average_precision_score(y_valid, scores),
            "roc_auc": roc_auc_score(y_valid, scores),
            "lift_at_10pct": top_k.loc[top_k["top_fraction"] == 0.10, "lift_at_k"].iloc[
                0
            ],
            "fit_seconds": fit_seconds,
            "score_seconds": score_seconds,
        }
    )
    validation_ranking_rows.append(top_k.assign(candidate=name))
    validation_scores[name] = scores
    validation_models[name] = pipeline

validation_results = pd.DataFrame(validation_rows).set_index("candidate")
validation_results.round(6)

,average_precision,roc_auc,lift_at_10pct,fit_seconds,score_seconds
candidate,,,,,
Baseline Logistic Regression,0.288234,0.688023,2.705123,0.797940,0.047838
Tuned Logistic Regression,0.282831,0.686609,2.643841,1.814516,0.039791
Tuned Random Forest,0.160494,0.552186,1.304412,1.130818,0.035036


In [18]:
validation_top_k = pd.concat(validation_ranking_rows, ignore_index=True)
validation_top_k[
    [
        "candidate",
        "top_fraction",
        "k",
        "captured_subscribers",
        "precision_at_k",
        "recall_at_k",
        "lift_at_k",
    ]
].round(6)

,candidate,top_fraction,k,captured_subscribers,precision_at_k,recall_at_k,lift_at_k
0,Baseline Logistic Regression,0.05,412,179,0.434466,0.156743,3.134090
1,Baseline Logistic Regression,0.10,824,309,0.375000,0.270578,2.705123
2,Baseline Logistic Regression,0.20,1648,499,0.302791,0.436953,2.184233
3,Tuned Logistic Regression,0.05,412,175,0.424757,0.153240,3.064055
4,Tuned Logistic Regression,0.10,824,302,0.366505,0.264448,2.643841
5,Tuned Logistic Regression,0.20,1648,493,0.299150,0.431699,2.157970
6,Tuned Random Forest,0.05,412,74,0.179612,0.064799,1.295657
7,Tuned Random Forest,0.10,824,149,0.180825,0.130473,1.304412
8,Tuned Random Forest,0.20,1648,296,0.179612,0.259194,1.295657


### 6.2 Results & Interpretation

Baseline AP `0.288234`가 tuned Logistic Regression `0.282831`, tuned Random Forest
`0.160494`보다 높다. Lift@10%도 각각 `2.705123`, `2.643841`, `1.304412`로 같은 순서이다.
Tuned Logistic Regression의 AP 감소 `0.005404`는 사전에 정한 0.005 허용 범위 밖이다.

Train 내부에서의 tuning 개선이 뒤쪽 Validation으로 이어지지 않았다.
현재 feature 정책과 탐색 범위에서는 baseline을 대체할 근거가 없다.
이는 모든 비선형 모델의 한계가 아니라 이번 시간 분할과 초기 설정·탐색에 대한 결과이다.

### 6.3 Temporal Stability & Limitations

Validation을 원본 순서의 네 구간으로 나누고 같은 점수에서 AP와 양성 비율을 확인한다.
추가 학습이나 모델 재선택에 사용하지 않는 진단이며, 동일한 달력상 기간을 뜻하지 않는다.

In [19]:
block_rows = []
for block, positions in enumerate(np.array_split(np.arange(len(y_valid)), 4), start=1):
    block_y = y_valid.iloc[positions]
    for name, scores in validation_scores.items():
        block_rows.append(
            {
                "block": block,
                "candidate": name,
                "rows": len(positions),
                "prevalence": block_y.mean(),
                "average_precision": average_precision_score(
                    block_y, scores[positions]
                ),
            }
        )
validation_blocks = pd.DataFrame(block_rows)
validation_blocks.round(6)

,block,candidate,rows,prevalence,average_precision
0,1,Baseline Logistic Regression,2060,0.182524,0.248350
1,1,Tuned Logistic Regression,2060,0.182524,0.249822
2,1,Tuned Random Forest,2060,0.182524,0.276568
3,2,Baseline Logistic Regression,2060,0.057282,0.108697
4,2,Tuned Logistic Regression,2060,0.057282,0.108271
5,2,Tuned Random Forest,2060,0.057282,0.072727
6,3,Baseline Logistic Regression,2059,0.079650,0.123897
7,3,Tuned Logistic Regression,2059,0.079650,0.127166
8,3,Tuned Random Forest,2059,0.079650,0.091239
9,4,Baseline Logistic Regression,2059,0.235066,0.386684


Baseline의 구간별 AP는 `0.108697`~`0.386684`, 양성 비율은 약 5.73~23.51%이다.
성능 변동을 모델 불안정성만으로 해석할 수 없다. Random Forest는 첫 구간에서 앞서지만
나머지 구간과 전체 Validation에서는 뒤진다. 하나의 후기 기간으로 미래 안정성을 보장할 수 없다.

## 7. Model Selection & Final Configuration

AP 허용 범위와 Lift 기준을 만족하는 후보 중 단순한 구성을 선택한다.
단순성 우선순위는 baseline Logistic Regression, tuned Logistic Regression,
tuned 비선형 모델 순서이며, 허용 범위는 통계적 유의성을 뜻하지 않는다.
선택된 모델의 전체 파라미터와 실행 환경을 아래에 기록한다.

In [20]:
ap_tolerance = 0.005
lift_relative_tolerance = 0.02
eligible = validation_results.loc[
    validation_results["average_precision"]
    >= validation_results["average_precision"].max() - ap_tolerance
]
eligible = eligible.loc[
    eligible["lift_at_10pct"]
    >= eligible["lift_at_10pct"].max() * (1 - lift_relative_tolerance)
]
simplicity_order = [name for name, _ in validation_candidates]
selected_name = next(name for name in simplicity_order if name in eligible.index)
selected_pipeline = clone(validation_models[selected_name])
frozen_model_parameters = selected_pipeline.named_steps["model"].get_params()

print("Selected candidate:", selected_name)
print("Frozen estimator:", selected_pipeline.named_steps["model"])
print("Frozen parameters:", frozen_model_parameters)
validation_results.loc[[selected_name]].round(6)

Selected candidate: Baseline Logistic Regression
Frozen estimator: LogisticRegression(max_iter=1000)
Frozen parameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


,average_precision,roc_auc,lift_at_10pct,fit_seconds,score_seconds
candidate,,,,,
Baseline Logistic Regression,0.288234,0.688023,2.705123,0.79794,0.047838


In [21]:
import platform
from importlib.metadata import version

pd.Series(
    {
        "python": platform.python_version(),
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "scikit-learn": version("scikit-learn"),
        "scipy": version("scipy"),
    },
    name="version",
)

python          3.12.15
numpy             2.5.3
pandas            3.0.6
scikit-learn      1.9.1
scipy            1.18.1
Name: version, dtype: str

### 7.1 Frozen Evaluation Protocol

2026-10-26 Test 접근 전에 아래 구성을 확정했다. 이번 편집에서는 모델과 평가 규칙을 변경하지 않았다.

| 항목 | 확정 내용 |
|---|---|
| 입력 | `duration`과 `y`를 제외한 기존 원본 예측 변수 19개 |
| 전처리 | 일반 수치형 8개 scaling, sentinel을 분리한 `pdays` 2개 scaling, 범주형 10개 one-hot; `unknown` 유지 |
| 모델 | `LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, tol=0.0001)`; 나머지는 위 파라미터 출력과 scikit-learn 1.9.1 환경 유지 |
| 최종 학습 | 새 전체 Pipeline을 Train+Validation 37,069행에 fit |
| 최종 평가 대상 | 원본 나머지 4,119행; 셔플·재정렬 없음 |
| 점수와 지표 | 반올림하지 않은 양성 클래스 점수로 AP, ROC-AUC, 양성 비율과 Top-5/10/20% Precision·Recall·Lift |
| Top-K | `ceil(n*p)`, 내림차순 stable sort, 원본 순서로 동점 처리, 평가 대상 양성 비율 기준 Lift |
| Test 사용 | 확정한 모델 하나를 한 번 평가하며 결과로 tuning·재선택하지 않음 |

이 노트북의 개발에는 Test를 사용하지 않았다. 과거 EDA에서 Test target 집계 정보를 확인한
이력은 유지한다. 최종 평가 결과는 [Final Evaluation](05-final-evaluation.ipynb)에 별도로 기록한다.

## 8. Experiment Summary

| 항목 | 결과 |
|---|---|
| 선택 모델 | Sentinel-aware Logistic Regression baseline |
| Validation AP | 0.288234 |
| Validation ROC-AUC | 0.688023 |
| Precision@10% | 0.375000 |
| Recall@10% | 0.270578 |
| Lift@10% | 2.705123 |

초기 모델 비교와 제한된 tuning은 Validation 개선으로 이어지지 않았다.
기존 baseline이 더 높은 AP와 Lift를 보였으므로 단순한 구성을 유지한다.
상위 10%의 824개 레코드에 가입 레코드 309개를 포함하지만, 수익성이나 연락의 인과효과는 판단하지 않는다.

시간에 따른 양성 비율과 feature 분포 변화, 고객 식별자와 완전한 시각 정보의 부재는
평가의 한계로 남는다. 확률 보정이나 운영 연락 용량을 검증한 결과가 아니며,
확정한 구성의 최종 일반화 성능은 별도 Test 평가로 확인한다.